# Customer Churn Prediction: Experiment Notebook

Follows `methodology.md`. Dataset: IBM Telco Customer Churn (7,043 customers, 21 columns).

**Stage 1:** load and clean the data.
**Stage 2:** engineer features and split the data.
**Stage 3:** train the models.
**Stage 4:** evaluate on the held-out test set.
**Stage 5:** findings.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve,
                             precision_recall_curve)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load the data
Try the public IBM copy first. If there is no internet access, upload `Telco-Customer-Churn.csv` manually.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(URL)
except Exception as e:
    print("Download failed:", e)
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))

print(df.shape)
df.head()

In [ ]:
df.info()
print()
print("Churn distribution:")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))

## 2. Clean the data
Cleaning plan from `methodology.md`:
1. Convert `TotalCharges` from text to numeric. Blank strings become NaN.
2. Fill those NaN values with 0. They belong to customers with `tenure == 0` (brand new, nothing billed yet).
3. Drop `customerID` (identifier, no predictive meaning).
4. Encode the target: Yes = 1, No = 0.
5. Collapse "No internet service" and "No phone service" into "No", since they duplicate information in `InternetService` and `PhoneService`.
6. Check for duplicates and remaining missing values.

In [ ]:
df = df.copy()

# 1-2. TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Rows with missing TotalCharges:", df["TotalCharges"].isna().sum())
print("Their tenure values:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# 3. Drop identifier
df = df.drop(columns=["customerID"])

# 4. Target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)

# 5. Collapse redundant categories
df = df.replace({"No internet service": "No", "No phone service": "No"})

# 6. Checks
print("Duplicate rows:", df.duplicated().sum())
print("Missing values left:", int(df.isna().sum().sum()))
df.shape

## 3. Feature engineering
New features, all computed from the customer's own row (no use of the target, so no leakage):
- `tenure_group`: tenure binned into 0-12, 13-24, 25-48, 49-72 months
- `num_services`: count of add-on services the customer has
- `avg_monthly_spend`: TotalCharges / tenure (0 when tenure is 0)
- `charge_gap`: MonthlyCharges minus avg_monthly_spend, i.e. whether the customer's price has risen
- `is_month_to_month`: 1 if on a month-to-month contract
- `auto_pay`: 1 if payment method is automatic (bank transfer or credit card)

In [ ]:
def add_features(data):
    d = data.copy()

    d["tenure_group"] = pd.cut(d["tenure"], bins=[-1, 12, 24, 48, 72],
                               labels=["0-12", "13-24", "25-48", "49-72"]).astype(str)

    service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                    "TechSupport", "StreamingTV", "StreamingMovies"]
    d["num_services"] = (d[service_cols] == "Yes").sum(axis=1)

    d["avg_monthly_spend"] = np.where(d["tenure"] > 0, d["TotalCharges"] / d["tenure"], 0)
    d["charge_gap"] = d["MonthlyCharges"] - d["avg_monthly_spend"]

    d["is_month_to_month"] = (d["Contract"] == "Month-to-month").astype(int)
    d["auto_pay"] = d["PaymentMethod"].str.contains("automatic", case=False).astype(int)
    return d

df_fe = add_features(df)
df_fe[["tenure", "tenure_group", "num_services", "avg_monthly_spend",
       "charge_gap", "is_month_to_month", "auto_pay"]].head()

## 4. Train / test split and preprocessing
Stratified 80/20 split. The test set is not touched until final evaluation.
Scaling and one-hot encoding live inside a `Pipeline`, so they are fit only on training data.

In [ ]:
X = df_fe.drop(columns="Churn")
y = df_fe["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape,  "| churn rate:", round(y_test.mean(), 3))

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges",
                "num_services", "avg_monthly_spend", "charge_gap"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
])

## 5. Train the models
Three models, as planned in `methodology.md`:
1. **Logistic Regression**: interpretable baseline
2. **Random Forest**: non-linear, robust
3. **XGBoost**: gradient boosting, usually strongest on tabular data

Class imbalance (about 26% churn) is handled with class weights rather than SMOTE. Each model is compared with 5-fold stratified cross-validation on the training set.

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced",
                                              random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=neg / pos, eval_metric="logloss",
                             random_state=RANDOM_STATE),
}

pipelines = {name: Pipeline([("prep", preprocess), ("model", m)]) for name, m in models.items()}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "recall": "recall", "precision": "precision", "f1": "f1"}

cv_rows = []
for name, pipe in pipelines.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({"model": name,
                    **{k: res[f"test_{k}"].mean() for k in scoring},
                    "roc_auc_std": res["test_roc_auc"].std()})

cv_results = pd.DataFrame(cv_rows).set_index("model").round(3)
cv_results

### Decision threshold
With class weights, 0.5 is not necessarily the best cut-off. I choose, for each model, the threshold that maximizes F1 on **out-of-fold** predictions from the training set. The test set plays no part in this choice.

In [ ]:
best_threshold = {}
for name, pipe in pipelines.items():
    oof = cross_val_predict(pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    prec, rec, thr = precision_recall_curve(y_train, oof)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_threshold[name] = float(thr[np.argmax(f1s)])

pd.Series(best_threshold, name="chosen threshold").round(3)

## 6. Final evaluation on the test set
Metrics from `methodology.md`:
- **ROC-AUC**: main metric, threshold-independent ranking quality
- **PR-AUC**: better than ROC-AUC when the positive class is a minority
- **Recall, precision, F1 on the churn class**: tied to the business goal of catching customers who will leave

In [ ]:
fitted = {}
rows = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= best_threshold[name]).astype(int)
    rows.append({
        "model": name,
        "threshold": round(best_threshold[name], 3),
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "recall": recall_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    })

test_results = pd.DataFrame(rows).set_index("model").round(3)
test_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, pipe in fitted.items():
    proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, proba):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=f"{name} (AP {average_precision_score(y_test, proba):.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set(title="ROC curve (test set)", xlabel="False positive rate", ylabel="True positive rate")
axes[1].axhline(y_test.mean(), color="k", ls="--", alpha=0.4)
axes[1].set(title="Precision-recall curve (test set)", xlabel="Recall", ylabel="Precision")
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipe) in zip(axes, fitted.items()):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= best_threshold[name]).astype(int)
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Stay", "Churn"], yticklabels=["Stay", "Churn"], ax=ax)
    ax.set(title=name, xlabel="Predicted", ylabel="Actual")
plt.tight_layout(); plt.show()

In [ ]:
# What drives churn? Logistic regression coefficients and XGBoost importances.
feature_names = fitted["XGBoost"].named_steps["prep"].get_feature_names_out()

xgb_imp = pd.Series(fitted["XGBoost"].named_steps["model"].feature_importances_,
                    index=feature_names).sort_values(ascending=False).head(12)

lr_coef = pd.Series(fitted["Logistic Regression"].named_steps["model"].coef_[0],
                    index=feature_names)
lr_top = lr_coef.reindex(lr_coef.abs().sort_values(ascending=False).head(12).index)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xgb_imp[::-1].plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("XGBoost: top 12 feature importances")
lr_top[::-1].plot.barh(ax=axes[1], color=["tomato" if v > 0 else "seagreen" for v in lr_top[::-1]])
axes[1].set_title("Logistic Regression: top 12 coefficients\n(red raises churn risk, green lowers it)")
plt.tight_layout(); plt.show()

## 7. Findings and what I would change

> **TODO: fill this in after running the notebook (5-6 sentences).**
> Suggested structure:
> 1. Which model performed best on ROC-AUC and PR-AUC, and by how much over the baseline.
> 2. Whether the tree models clearly beat logistic regression or the gap was small.
> 3. The recall / precision trade-off at the chosen threshold, in business terms (how many churners caught vs. how many false alarms).
> 4. The top features driving churn (for example contract type, tenure, monthly charges).
> 5. Main limitation (single snapshot, no cost data, no time dimension).
> 6. What you would change with more time (hyperparameter tuning, SMOTE comparison, calibration, SHAP, a cost-based threshold, a time-based split on real data).